# Scraper for 2025 H2H Shoots

Run the code cell below to import all rows in the **Shoots** sheet where **Imported = 0**.

In [4]:
from pathlib import Path
import re
import sys

import pandas as pd
from openpyxl import load_workbook

# Resolve paths whether kernel cwd is repo root or the notebook folder.
CWD = Path.cwd()
if (CWD / "main.py").exists() and (CWD / "2025 H2Hs").exists():
    PROJECT_ROOT = CWD
    NOTEBOOK_DIR = CWD / "2025 H2Hs"
else:
    NOTEBOOK_DIR = CWD
    PROJECT_ROOT = CWD.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from main import _clean_scraped_table  # Reuse existing table-cleaning logic.

WORKBOOK_PATH = NOTEBOOK_DIR / "2025 H2H Shoots.xlsx"
if not WORKBOOK_PATH.exists():
    raise FileNotFoundError(f"Workbook not found: {WORKBOOK_PATH}")
SHOOTS_SHEET = "Shoots"


def _find_column(columns, candidate_names):
    lowered = {str(col).strip().lower(): col for col in columns}
    for name in candidate_names:
        if name.lower() in lowered:
            return lowered[name.lower()]
    raise KeyError(f"Missing required column. Expected one of: {candidate_names}")


def _clean_club(country_value):
    if pd.isna(country_value):
        return pd.NA
    text = str(country_value).strip()
    return re.sub(r"^\s*\d+\s*-\s*", "", text).strip()


def _scrape_event_table(url):
    tables = pd.read_html(url, flavor="lxml")
    if not tables:
        raise ValueError(f"No HTML table found at {url}")

    cleaned = _clean_scraped_table(tables[0])
    club_source_col = _find_column(
        cleaned.columns,
        ["Country", "Country or State Code"],
    )
    required = ["Pos.", "Athlete", "Tot.", club_source_col]
    missing = [col for col in required if col not in cleaned.columns]
    if missing:
        raise KeyError(f"Missing expected columns on page {url}: {missing}")

    result = cleaned.loc[:, required].copy()
    result["Club"] = result[club_source_col].apply(_clean_club)
    result = result.rename(
        columns={
            "Pos.": "Rank",
            "Athlete": "Name",
            "Tot.": "Score",
        }
    )
    return result[["Rank", "Name", "Club", "Score"]]


# Load workbook metadata from Shoots sheet.
shoots_df = pd.read_excel(WORKBOOK_PATH, sheet_name=SHOOTS_SHEET)
url_col = _find_column(shoots_df.columns, ["URL", "Url"])
event_col = _find_column(shoots_df.columns, ["Event Name", "Event"])
imported_col = _find_column(shoots_df.columns, ["Imported", "imported"])

pending_mask = shoots_df[imported_col].fillna(0).astype(int) == 0
pending_rows = shoots_df[pending_mask].copy()

if pending_rows.empty:
    print("No rows with Imported = 0 were found.")
else:
    wb = load_workbook(WORKBOOK_PATH)
    processed = []
    errors = []

    for row_index, row in pending_rows.iterrows():
        event_name = str(row[event_col]).strip()
        url = str(row[url_col]).strip()

        if not event_name or event_name.lower() == "nan":
            errors.append(f"Row {row_index}: missing Event Name")
            continue
        if not url or url.lower() == "nan":
            errors.append(f"Row {row_index}: missing URL")
            continue

        try:
            event_df = _scrape_event_table(url)

            if event_name in wb.sheetnames:
                del wb[event_name]
            ws = wb.create_sheet(title=event_name)

            ws.append(list(event_df.columns))
            for values in event_df.itertuples(index=False, name=None):
                ws.append(list(values))

            shoots_df.at[row_index, imported_col] = 1
            processed.append((event_name, len(event_df)))
        except Exception as exc:
            errors.append(f"{event_name}: {exc}")

    # Rewrite Shoots sheet with updated Imported flags.
    if SHOOTS_SHEET in wb.sheetnames:
        del wb[SHOOTS_SHEET]
    ws_shoots = wb.create_sheet(title=SHOOTS_SHEET, index=0)
    ws_shoots.append(shoots_df.columns.tolist())
    for values in shoots_df.itertuples(index=False, name=None):
        ws_shoots.append(list(values))

    wb.save(WORKBOOK_PATH)

    print(f"Processed events: {len(processed)}")
    for event_name, rows_written in processed:
        print(f"  - {event_name}: {rows_written} rows")

    if errors:
        print("\nEvents with errors:")
        for item in errors:
            print(f"  - {item}")

Processed events: 8
  - Long Mynd: 8 rows
  - AGB NT S1: 119 rows
  - AGB NT S2: 105 rows
  - Eccles: 19 rows
  - DPA: 24 rows
  - AGB NT S3: 112 rows
  - AGB NT S5: 59 rows
  - Andover: 28 rows

Events with errors:
  - Bowmen of Glen May Saturday: HTTP Error 404: Not Found
  - Llantarnam: HTTP Error 404: Not Found
  - Meriden Summer: <urlopen error [SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1077)>
  - Meriden Mid-Summer: HTTP Error 404: Not Found
